In [ ]:
"""
Sweep the Kerr strength chi and plot, for each chi, the period-averaged
Hellinger distance between

  * Q_HB4   : the FOURTH-CUMULANT (Edgeworth-corrected Gaussian) Husimi
              function rebuilt from the harmonic-balance moment solution of
              hhb/kerr/hb_moments_kerr_n.py at order=4 (14 real DOFs: mu,
              kappa_20, kappa_11, kappa_30, kappa_21, kappa_40, kappa_31,
              kappa_22),
  * Q_exact : the Husimi function of the truncated-Fock Lindblad steady state
              obtained with dynamiqs.

This is sweep_kerr_hellinger_3.ipynb with (i) the solver swapped for the
arbitrary-order module at order=4 and (ii) the Edgeworth correction extended
by the fourth-cumulant term.

Q_HB4 is the Gram-Charlier / first-order-Edgeworth reconstruction of the
density whose cumulant-generating function the closure truncates:

    Q_4(z) = Q_G(z) * [ 1 + sum_{3 <= j+k <= 4} kappa_jk/(j! k!) * h_jk(z) ]
           = Q_G(z) * [ 1 + 2 Re( k30/6  h30 + k21/2 h21 )
                          + 2 Re( k40/24 h40 + k31/6 h31 ) + k22/4 h22 ]

with h_jk = (-d/dZ)^j (-d/dZ*)^k Q_G / Q_G (Wirtinger derivatives, Z = z-mu);
the (j,k) and (k,j) terms are complex conjugates of each other, hence the
2 Re(...), while the self-conjugate (2,2) term is counted once and is real.

Why the *linear* (Gram-Charlier) form and not a re-exponentiated Edgeworth
series: transforming term by term, chi(s,u) = chi_G(s,u) * (1 + P(s,u)) with
P the degree-3+4 polynomial above, so log chi = K_G + P - P^2/2 + ... and
P^2 only has degree >= 6. The reconstruction therefore reproduces cumulants
1..4 EXACTLY (checked numerically at 1e-13: norm 1, first moments 0, second
cumulants untouched, kappa_30/21/40/31/22 recovered) and pollutes only orders
>= 5, which the closure does not track anyway.

The derivatives, with phi = log Q_G + const, D = sig^2 - |sigt|^2:
    g   = dphi/dZ = (sigt* Z - sig Z*)/D,   gb = conj(g)
    A   = dg/dZ   = sigt*/D,  Ab = conj(A) = sigt/D,  B = dg/dZ* = -sig/D
    h30 = -(g^3 + 3 g A)
    h21 = -(g^2 gb + A gb + 2 g B)
    h40 =   g^4 + 6 g^2 A + 3 A^2
    h31 =   g^3 gb + 3 g^2 B + 3 g A gb + 3 A B
    h22 =   g^2 gb^2 + A gb^2 + Ab g^2 + 4 g gb B + A Ab + 2 B^2   (real)
(the order-3 h's carry the (-1)^(j+k) sign, the order-4 h's do not).

Caveat, worse at order 4 than at order 3: the quartic polynomial factor makes
the Edgeworth tails dip further below zero, so Q_4 is clipped at 0 exactly as
in the order-3 notebook (and the Hellinger integrand clips too). Clipping
breaks normalisation slightly, so the sweep also records the mean clipped
norm  int Q_4 dA  as a diagnostic -- read a large departure from 1 as "the
Gram-Charlier reconstruction, not the closure, is what is failing here".

Range of chi: the sweep stops at chi = 1.2. Beyond that the order-4 Newton
solve stops making progress on this continuation ladder (first failure at
chi ~ 1.33-1.34, with N_H = 14; a 10x finer ladder does not get past it),
which is the fourth-cumulant DOFs -- not the Fock reference -- running out of
road as the Q function goes bimodal.

Run:  python3 sweep_kerr_hellinger_4.py
"""

import numpy as np
import matplotlib.pyplot as plt

# Artefact locations resolve from the installed package, not the working
# directory: data/kerr/*.npz is the tracked regression baseline.
from hhb.paths import KERR_DATA_DIR as DATA_DIR
DATA_DIR.mkdir(parents=True, exist_ok=True)

from hhb.kerr.hb_moments_kerr_n import (
    KerrParams, HBSettingsMomentsN, MomentHillMethodN)

import dynamiqs as dq
dq.set_precision('double')
import jax.numpy as jnp


# ----------------------------------------------------------------------
# Fixed physical parameters (notebook values); chi is the swept quantity
# ----------------------------------------------------------------------
Delta = 1.0
kappa = 1.0
eps = 3.0
omega_d = 1.3

# HB settings
ORDER = 4                   # cumulant truncation degree -> 14 real DOFs
N_H = 14
SAMPLES_PER_HARMONIC = 48

# Fock / dynamiqs settings
N_FOCK = 20
N_PERIODS_TOTAL = 20        # transient + steady state
N_PERIODS_USE = 2           # periods kept for the phase average
N_SAVE = 4000

# phase-space grid for the Hellinger integral
GRID_LIM = 10.0
GRID_PTS = 100


# ----------------------------------------------------------------------
# Husimi functions
# ----------------------------------------------------------------------
def Q_hb_grid(X, Y, mu, sig, sigt):
    """Gaussian Q function; sig = sigma^2 = <da da^dag> (anti-normal), sigt = <da^2>."""
    Z = (X + 1j * Y) - mu
    denom = sig**2 - np.abs(sigt)**2
    return (1 / np.pi / np.sqrt(denom)
            * np.exp((-sig * np.abs(Z)**2 + np.real(np.conj(sigt) * Z**2)) / denom))


def Q_hb_grid_4(X, Y, mu, sig, sigt, k30, k21, k40=0.0, k31=0.0, k22=0.0):
    """Edgeworth-corrected Gaussian Q carrying the third AND fourth cumulants.

    Q_4 = Q_G * [1 + 2 Re(k30/6 h30 + k21/2 h21)
                   + 2 Re(k40/24 h40 + k31/6 h31) + k22/4 h22]

    See the module docstring for h30..h22. Setting the fourth cumulants to
    zero recovers Q_hb_grid_3 of sweep_kerr_hellinger_3.ipynb exactly, and
    setting all of them to zero recovers the Gaussian Q_hb_grid.
    Clipped at 0 (Edgeworth tails can dip negative).
    """
    Z = (X + 1j * Y) - mu
    D = sig**2 - np.abs(sigt)**2
    QG = (1 / np.pi / np.sqrt(D)
          * np.exp((-sig * np.abs(Z)**2 + np.real(np.conj(sigt) * Z**2)) / D))
    g = (np.conj(sigt) * Z - sig * np.conj(Z)) / D
    gb = np.conj(g)
    A = np.conj(sigt) / D
    Ab = np.conj(A)
    B = -sig / D                      # real
    # order 3: h_jk = (-d_Z)^j (-d_Z*)^k Q_G / Q_G, so an overall minus sign
    h30 = -(g**3 + 3 * g * A)
    h21 = -(g**2 * gb + A * gb + 2 * g * B)
    # order 4: (-1)^(j+k) = +1
    h40 = g**4 + 6 * g**2 * A + 3 * A**2
    h31 = g**3 * gb + 3 * g**2 * B + 3 * g * A * gb + 3 * A * B
    h22 = (g**2 * gb**2 + A * gb**2 + Ab * g**2
           + 4 * g * gb * B + A * Ab + 2 * B**2)      # real up to roundoff
    corr = (1
            + 2 * np.real(k30 / 6 * h30 + k21 / 2 * h21)
            + 2 * np.real(k40 / 24 * h40 + k31 / 6 * h31)
            + np.real(k22) / 4 * np.real(h22))
    return np.clip(QG * corr, 0.0, None)


def coherent_overlaps(alpha_grid, n_fock):
    n = np.arange(n_fock)
    log_norm = -0.5 * np.abs(alpha_grid)[:, None]**2 - 0.5 * np.array(
        [np.sum(np.log(np.arange(1, k + 1))) for k in n])[None, :]
    return np.exp(log_norm) * alpha_grid[:, None]**n[None, :]   # (M, n_fock)


def make_Q_exact(C, shape):
    def Q_exact_grid(rho):
        rho = np.asarray(rho)
        Q_flat = np.real(np.einsum('mi,ij,mj->m', C.conj(), rho, C)) / np.pi
        return Q_flat.reshape(shape)
    return Q_exact_grid


def hellinger(Q1, Q2, dA):
    return np.sqrt(max(0.0, 1 - np.sum(np.sqrt(np.clip(Q1 * Q2, 0, None))) * dA))


def make_fourier_interp(samples, T):
    """samples: 1D array (real or complex) uniformly sampled on [0, T)."""
    N = samples.size
    coeffs = np.fft.fft(samples) / N
    freqs = np.fft.fftfreq(N, d=T / N) * 2 * np.pi
    def f_at(t):
        t = np.atleast_1d(t)
        return np.exp(1j * np.outer(t, freqs)) @ coeffs
    return f_at


# ----------------------------------------------------------------------
# The two solvers, one chi at a time
# ----------------------------------------------------------------------
def solve_hb(chi, z_guess=None):
    """Order-4 HB solve. The symbolic derivation is cached per process, so
    only the first chi pays for it (~30 s)."""
    kerr = KerrParams(Delta=Delta, chi=chi, kappa=kappa, eps=eps, omega_d=omega_d)
    settings = HBSettingsMomentsN(order=ORDER, N_H=N_H,
                                  samples_per_harmonic=SAMPLES_PER_HARMONIC)
    hb = MomentHillMethodN(kerr, settings)

    # NB: the _n module's packing is graded (mu, k20, k11, k30, ...), i.e.
    # sigma^2 is NOT at index 2 as in the _3 module -- use index_map.
    def fresh_guess():
        z0 = np.zeros(hb.dim)
        z0[hb.index_map['k11'][0]] = np.sqrt(2) * 1.0   # DC sigma^2 = 1 (vacuum)
        return z0

    guesses = [z_guess, fresh_guess()] if z_guess is not None else [fresh_guess()]
    last = None
    for z0 in guesses:
        try:
            return hb, hb.solve(z0)
        except RuntimeError as err:                      # continuation failed
            last = err
    raise last


def solve_exact(chi):
    """Lindblad evolution; returns (T, t_use, states_use)."""
    a = dq.destroy(N_FOCK)
    H_0 = Delta * a.dag() @ a + chi / 2 * (a.dag() @ a.dag()) @ (a @ a)
    H_drive_a = dq.modulated(lambda t: eps / 2 * (1 + jnp.exp(-2j * omega_d * t)), a)
    H_drive_a_dag = dq.modulated(lambda t: eps / 2 * (1 + jnp.exp(2j * omega_d * t)), a.dag())
    H_tot = H_0 + H_drive_a + H_drive_a_dag
    jump_op = [jnp.sqrt(kappa) * a]

    T = 2 * np.pi / (2 * omega_d)     # folding period of the drive
    t_max = N_PERIODS_TOTAL * T
    t_save = np.linspace(0, t_max, N_SAVE)

    psi_0 = dq.coherent(N_FOCK, -2)
    method = dq.method.Tsit5(atol=1e-12, rtol=1e-12)
    res = dq.mesolve(H_tot, jump_op, psi_0, t_save, method=method)

    mask = t_save >= t_max - N_PERIODS_USE * T
    idx = np.where(mask)[0]
    return T, t_save[idx], [res.states[i] for i in idx]


def mean_hellinger_for_chi(chi, X, Y, Q_exact_grid, dA, z_guess=None):
    hb, z_sol = solve_hb(chi, z_guess)
    rec = hb.reconstruct(z_sol)          # time series on hb.tau_j, keyed by name

    T, t_use, states_use = solve_exact(chi)

    # HB interpolants live on hb.T_fund; evaluating at absolute times is safe.
    def interp(name, real=False):
        f = make_fourier_interp(rec[name], hb.T_fund)(t_use)
        return np.real(f) if real else f

    mu_t = interp('mu')
    sig_t = interp('k11', real=True)     # sigma^2  (anti-normal, -> 1 in vacuum)
    sigt_t = interp('k20')               # <da^2>
    k30_t, k21_t = interp('k30'), interp('k21')
    k40_t, k31_t = interp('k40'), interp('k31')
    k22_t = interp('k22', real=True)

    # Gaussian-part physicality: with n = <da+ da> = sigma^2 - 1 and m = <da^2>,
    # a single-mode Gaussian is physical iff n(n+1) >= |m|^2. (sigma^2 >= 1 + |m|
    # is too strict -- it flags the exact Lindblad state as unphysical.)
    n_occ = sig_t - 1.0
    viol = n_occ * (n_occ + 1.0) < np.abs(sigt_t)**2 - 1e-9
    n_viol = int(viol.sum())

    distances = np.empty(len(states_use))
    norms = np.empty(len(states_use))
    for k, rho_k in enumerate(states_use):
        Q4 = Q_hb_grid_4(X, Y, mu_t[k], sig_t[k], sigt_t[k],
                         k30_t[k], k21_t[k], k40_t[k], k31_t[k], k22_t[k])
        norms[k] = Q4.sum() * dA                  # 1 minus the clipped tails
        distances[k] = hellinger(Q_exact_grid(rho_k), Q4, dA)

    # fold onto one period and average with the trapezoid rule
    t_mod = t_use % T
    order = np.argsort(t_mod)
    avg = np.trapezoid(distances[order], t_mod[order]) / T

    return avg, distances.max(), norms.mean(), n_viol, len(distances), z_sol


In [ ]:
real_axis = np.linspace(-GRID_LIM, GRID_LIM, GRID_PTS)
imag_axis = np.linspace(-GRID_LIM, GRID_LIM, GRID_PTS)
X, Y = np.meshgrid(real_axis, imag_axis)
dA = (real_axis[1] - real_axis[0]) * (imag_axis[1] - imag_axis[0])

C = coherent_overlaps((X + 1j * Y).ravel(), N_FOCK)
Q_exact_grid = make_Q_exact(C, X.shape)

avg_list, max_list, norm_list = [], [], []
z_guess = None
CHI_VALUES = np.linspace(0.0, 1.2, 10)   # order-4 HB stalls above ~1.33
for chi in CHI_VALUES:
    try:
        avg, dmax, qnorm, n_viol, n_pts, z_guess = mean_hellinger_for_chi(
            chi, X, Y, Q_exact_grid, dA, z_guess
        )
    except RuntimeError as err:
        # keep the sweep going and restart the continuation from vacuum
        print(f"chi = {chi:.4f}   HB solve failed: {err}")
        avg_list.append(np.nan)
        max_list.append(np.nan)
        norm_list.append(np.nan)
        z_guess = None
        continue
    avg_list.append(avg)
    max_list.append(dmax)
    norm_list.append(qnorm)
    note = f"  [unphysical at {n_viol}/{n_pts} phases]" if n_viol else ""
    print(f"chi = {chi:.4f}   mean Hellinger = {avg:.4f}   max = {dmax:.4f}"
          f"   <norm Q4> = {qnorm:.4f}{note}")

avg_arr = np.array(avg_list)
max_arr = np.array(max_list)
norm_arr = np.array(norm_list)
np.savez(DATA_DIR / 'sweep_kerr_hellinger_4.npz',
         chi=CHI_VALUES, mean=avg_arr, max=max_arr, norm=norm_arr)

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(CHI_VALUES, avg_arr, 'o-', label='period-averaged (4th order)')
plt.plot(CHI_VALUES, max_arr, 's--', color='C0', alpha=0.7, label='worst phase')
for fname, colour, lab in (('sweep_kerr_hellinger_3.npz', 'C2', '3rd order'),
                           ('sweep_kerr_hellinger.npz', 'C3', 'Gaussian')):
    try:
        ref = np.load(DATA_DIR / fname)
    except FileNotFoundError:
        continue
    plt.plot(ref['chi'], ref['mean'], 'o-', color=colour, alpha=0.6,
             label=f'period-averaged ({lab})')
    plt.plot(ref['chi'], ref['max'], 's--', color=colour, alpha=0.6,
            label=f'max ({lab})')
plt.xlabel(r'Kerr strength $\chi$')
plt.ylabel('Hellinger distance')
plt.title(r'$Q_{\rm HB4}$ vs $Q_{\rm exact}$, averaged over one drive period')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Clipped-norm diagnostic: how much of Q_4 the Edgeworth tails cost us.
plt.figure(figsize=(6, 3))
plt.plot(CHI_VALUES, norm_arr, 'o-')
plt.axhline(1.0, color='k', lw=0.8, alpha=0.5)
plt.xlabel(r'Kerr strength $\chi$')
plt.ylabel(r'$\int Q_4\,d^2z$ (clipped)')
plt.title('Gram-Charlier reconstruction quality')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
